In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import nltk
from nltk.corpus import brown
from tensorflow import keras
from keras.layers import Embedding
from sklearn.model_selection import train_test_split


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Combine the words in each sentence of the Brown Corpus into a single string, 
# with each word separated by a space.
sentences = [" ".join(sentence) for sentence in brown.sents()]

# Extract the part-of-speech tags for each word in each sentence of the Brown Corpus.
tags = [[word[1] for  word in sentence] for sentence in brown.tagged_sents()]

# Combine the POS tags in each sentence into a single string, with each tag separated by a space.
tags = [" ".join(tag) for tag in tags]


In [3]:
# Split the dataset into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(sentences, tags,
                                                    test_size=0.09, random_state=42)

In [4]:
# Initialize a TextVectorization layer to preprocess and vectorize text data
word_vectorizer = keras.layers.TextVectorization( standardize = "lower",
                                                  output_sequence_length = 200 )
# Fit the vectorizer to the provided sentences to learn the vocabulary
word_vectorizer.adapt( X_train )

# 'vocabulary' contains all unique tokens learned during the vectorization process.
vocabulary = word_vectorizer.get_vocabulary()
# 'vocab_len' gives the total number of unique tokens in the vocabulary.
vocab_len = len(vocabulary)

# Create a dictionary mapping each word in the vocabulary to its index
word_index = dict(zip(vocabulary, range(vocab_len)))

In [5]:
# Initialize a TextVectorization layer to vectorize tags
tag_vectorizer = keras.layers.TextVectorization( standardize = None,
                                                 output_sequence_length = 200 )
# Fit the vectorizer to the provided tags 
tag_vectorizer.adapt( Y_train )

# 'tags' contains all unique tags learned during the vectorization process.
tags = tag_vectorizer.get_vocabulary()
# 'tags_len' gives the total number of unique tags in the vocabulary.
tags_len = len(tags)

In [6]:
# Download the GloVe Embeddings
if not os.path.exists('glove.6B.zip'):
    !wget https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
    !unzip -q glove.6B.zip

--2024-12-30 08:42:49--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... 

/usr/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  4.96MB/s    in 3m 11s  

2024-12-30 08:46:00 (4.32 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]



In [7]:
# Load GloVe word embeddings into a dictionary mapping words to their vector representations.
path_to_glove_file = "glove.6B.50d.txt"
embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print("Found %s word vectors." % len(embeddings_index))


Found 400000 word vectors.


In [8]:
embedding_dim = 50
hits = 0
misses = 0
missed_words = []

# Prepare embedding matrix
embedding_matrix = np.zeros((vocab_len, embedding_dim))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # Words not found in embedding index will be all-zeros.
        # This includes the representation for "padding" and "OOV"
        embedding_matrix[i] = embedding_vector
        hits += 1
    else:
        misses += 1
        missed_words.append(word)
print("Converted %d words (%d misses)" % (hits, misses))

Converted 39115 words (8693 misses)


In [9]:
missed_words[:10]

['',
 '[UNK]',
 "don't",
 "didn't",
 "it's",
 "i'm",
 "that's",
 "i'll",
 "can't",
 "couldn't"]

In [10]:
# Apply thevectorizer to transform the raw text data into numerical representations
X_train = word_vectorizer( X_train )
Y_train = tag_vectorizer( Y_train )

In [11]:
model = keras.Sequential([
    keras.Input(shape = (200,)),
    keras.layers.Embedding(weights = [embedding_matrix], input_dim = vocab_len,
                           output_dim = 50 ),
    keras.layers.Bidirectional(keras.layers.LSTM(units = 100, return_sequences = True )),
    keras.layers.Bidirectional(keras.layers.LSTM(units = 100, return_sequences = True)),
    keras.layers.TimeDistributed(keras.layers.Dense(units = tags_len, activation = "softmax")  )
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 200, 50)             │       2,390,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 200, 200)            │         120,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 200, 200)            │         240,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ (None, 200, 467)            │          93,867 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,845,867 (10.86 MB)

 Trainable params: 2,845,867 (10.86 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",  
    metrics=["accuracy"]
)
model.fit(X_train, Y_train, epochs = 10)

Epoch 1/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 138s 75ms/step - accuracy: 0.9424 - loss: 0.3661
Epoch 2/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.9929 - loss: 0.0276
Epoch 3/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.9951 - loss: 0.0174
Epoch 4/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.9962 - loss: 0.0131
Epoch 5/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.9969 - loss: 0.0104
Epoch 6/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.9974 - loss: 0.0085
Epoch 7/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.9977 - loss: 0.0074
Epoch 8/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.9981 - loss: 0.0061
Epoch 9/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 122s 75ms/step - accuracy: 0.9985 - loss: 0.0050
Epoch 10/10
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.9987 - loss: 0.0043


In [13]:
X_test = word_vectorizer( X_test )
Y_test = tag_vectorizer( Y_test )
model.evaluate(X_test, Y_test)

162/162 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.9946 - loss: 0.0232


[0.02363981492817402, 0.9945573806762695]